## Databricks Unity Gateway (AI Gateway):

**Overview**
* **Unity Gateway** (formerly/also referred to as **Mosaic AI Gateway**) is Databricks' centralized **AI control plane** for governing, routing, securing, and monitoring all AI traffic across your organization.
* Rather than letting applications and coding agents scatter API keys and unmonitored calls across multiple LLM vendors and tools, Unity Gateway places **every Model, Provider, MCP Server, Skill, and Agent behind Unity Catalog governance**.
* Every asset is registered as a **Unity Catalog securable** (`catalog.schema.name`) with standard `GRANT`/`REVOKE` permissions, rate limits, PII/PHI guardrails, traffic routing/fallbacks, and centralized cost & token observability.

## Part 1: The Problem & Core Value Proposition ("One Governed Front Door")

### Why Organizations Need Unity Gateway
As enterprises adopt LLMs, MCP tools, and AI coding agents (like Claude Code, Codex, or custom Databricks Agents), three major risks emerge:
1. **Credential & API Sprawl:** Developers hardcoding OpenAI, Anthropic, or Gemini API keys locally without central revocation or audit trails.
2. **Unpredictable Token Spend:** No visibility into which team, user, or coding agent is burning tokens or hitting runaway loops.
3. **Data Leakage & Ungoverned Tool Access:** Agents sending PII/sensitive data to external models or executing unapproved actions on Slack, GitHub, or databases.

### How Unity Gateway Solves This
* **Unified Control Plane:** Every AI asset lives in **Unity Catalog** (`catalog.schema.asset_name`) and is governed using the exact same `GRANT EXECUTE` permissions, tags, and lineage as Delta tables.
* **Unified API Surface:** Applications call a single governed gateway endpoint (`/ai-gateway/mlflow/v1/chat/completions` or native provider pass-throughs) while Unity Gateway handles authentication, guardrails, routing, and logging behind the scenes.

## Part 2: Tab 1 — Models (`Model Services`)

![part2_tab1_1795118747899.png](./part2_tab1_1795118747899.png "part2_tab1_1795118747899.png")

The **Models** tab is the central catalog of governed **Model Services** available in your workspace.

### Key Concepts & UI Elements
* **Model vs. Model Service:**
  * A **Model** is the underlying LLM or embedding asset at rest (versions, weights, metadata).
  * A **Model Service** is the governed serving endpoint that translates, routes, rate-limits, and logs inference requests to one or more models.
* **Built-in Foundation Model Services (`system.ai`):**
  * Databricks automatically ships governed model services under the `system.ai` catalog/schema—including *GPT OSS 120B* (`gpt-oss-120b`), *GPT OSS 20B*, *Qwen3.5 122B A10B*, *Qwen3 Next Instruct*, *Gemma 3 12B*, *Meta Llama 3.3 70B*, *Llama 4 Maverick*, and embedding models like *GTE Large (En)* and *Qwen3 Embedding*.
* **Custom Model Services (`+ Model` button):**
  * Create your own logical Model Service name (e.g., `prod.ai.customer-support-llm`) that decouples your application code from the underlying model.
  * **Traffic Splitting & Canary Rollouts:** Split traffic (e.g., 80% / 20%) across up to 5 model destinations to test new models safely.
  * **Automatic Failover / Fallbacks:** Define backup models (e.g., primary Databricks-hosted model $
ightarrow$ fallback external provider) so rate limits or outages never break production apps.
* **Columns in the Table:**
  * **Configuration Icons:** Quick visual indicators showing active **Rate Limits**, **Usage Tracking / Inference Tables**, and **Service Policies (Guardrails)**.
  * **Total tokens (7d):** Rolling 7-day token consumption per service (e.g., `16.4K` on `gpt-oss-120b`).
  * **Location:** Three-part Unity Catalog namespace (`system.ai` or custom `catalog.schema`).

## Part 3: Tab 2 — Providers (`Model Provider Services`)

![part3_tab2_i_1795118965924.png](./part3_tab2_i_1795118965924.png "part3_tab2_i_1795118965924.png")
![part3_tab2_ii_1795118994474.png](./part3_tab2_ii_1795118994474.png "part3_tab2_ii_1795118994474.png")

The **Providers** tab governs connections to **external third-party LLM providers** (OpenAI, Anthropic, Google Gemini, AWS Bedrock, Azure OpenAI, Cohere, or custom OpenAI-compatible endpoints).

### How Providers Work in Unity Gateway
* **Centralized Credential Vault:** Admins configure the external provider's API key or cloud credentials **once** inside Unity Catalog (e.g., `external_models.gemini.gemini_free_provider`). End users and applications **never see or store raw vendor API keys**.
* **Granular Access Control:** Admins grant `EXECUTE` permission on the provider service to specific groups or service principals—controlling who is allowed to call external models.
* **Unified Governance on External Traffic:** Even though the model runs outside Databricks, every request routed through a Unity Gateway Provider still receives:
  * **Rate Limiting** (requests/min or tokens/min per user/endpoint)
  * **PII/PHI Guardrails** (blocking or masking sensitive data *before* it leaves your network boundary)
  * **Spend & Token Attribution** (`Total tokens (7d)` and estimated USD spend in `system.ai_gateway.external_model_spend`)
* **`+ Provider` Button:** Onboard new external or custom model providers in seconds without writing custom proxy infrastructure.

## Part 4: Tab 3 — MCPs (`Model Context Protocol Servers`)

![part4_tab3_1795119208532.png](./part4_tab3_1795119208532.png "part4_tab3_1795119208532.png")

The **MCPs** tab makes Unity Gateway an enterprise **MCP Registry & Security Proxy**, governing how agents discover and invoke tools over the **Model Context Protocol (MCP)**.

### What You See in the MCPs Tab
1. **Featured & Managed MCP Servers (`system.ai`):**
   * **First-Party Databricks MCPs (`Active` out of the box):**
     * **`genie_one_mcp` (Genie One):** Lets agents ask natural-language questions against your structured & unstructured enterprise data via AI/BI Genie.
     * **`dbsql` (Databricks SQL):** Lets agents execute governed SQL queries on a Databricks SQL Warehouse.
   * **Pre-built Third-Party Enterprise MCPs (`Action needed` until OAuth/connection is configured):**
     * **`atlassian`:** Search Jira issues and Confluence pages.
     * **`github`:** Search code, manage issues, and inspect pull requests.
     * **`slack`:** Send and search messages across Slack channels.
     * **`gmail`, `google_calendar`, `google_drive`, `microsoft_365`:** Governed enterprise productivity connectors.
2. **Custom MCP Registration (`+ MCP` Button):**
   * Register your own internal MCP servers (hosted on **Databricks Apps** or external HTTP endpoints) as Unity Catalog securables.
3. **Tool-Level Governance & Identity Passthrough:**
   * Every MCP tool invocation inherits the calling user's permissions, supports tool allow/deny filtering, and writes a tamper-evident audit trail (`mcpCall` events in Unity Catalog audit logs).

## Part 5: Tab 4 — Skills (Governed Agent Playbooks)

![part5_1795119290921.png](./part5_1795119290921.png "part5_1795119290921.png")

The **Skills** tab introduces **Unity Catalog–governed Agent Skills**—modular, reusable instruction packages and reference files that agents load on demand to perform specialized workflows consistently.

### How Unity Gateway Skills Work
* **What is a Skill?** A structured folder (containing markdown instructions, scripts, templates, or domain guidelines) published into a Unity Catalog schema as a three-part securable (`catalog.schema.skill_name`).
* **Why Govern Skills in Unity Gateway?**
  * **Standardized Best Practices Across Coding Agents:** Instead of every developer maintaining ad-hoc local prompt files, platform teams publish approved skills (e.g., *"Databricks Medallion ETL Standards"*, *"Company Security Review Playbook"*, or *"SQL Performance Tuning Guide"*).
  * **Permission-Controlled Sharing:** Share skills across teams using standard Unity Catalog `GRANT` permissions.
  * **Auditability:** Unity Gateway logs skill listings, loads, and file fetches (`aiGatewaySkills` audit events) so organizations know which playbooks agents are using.
* **`+ Create skill` & `Enable skills for coding agents`:** Build new skills directly from the UI/CLI and connect them to developer coding agents.

## Part 6: Tab 5 — Agents (Agent Supervision & Workflows)

![part6_tab5_1795119353175.png](./part6_tab5_1795119353175.png "part6_tab5_1795119353175.png")

The **Agents** tab provides centralized visibility, registration, and runtime supervision for AI agents operating across your workspace.

### Key Capabilities in the Agents Tab
* **Central Agent Inventory:** Search and filter agents by **Agent Type** and **Owner**, tracking their lifecycle, permissions, and last modified timestamps in one place.
* **Omnigent / Harness Supervision:** Unity Gateway can sit above external coding harnesses (such as Claude Code, Codex, and custom agents) to enforce runtime safety policies, spend caps, and unified MLflow session tracing.
* **Quick-Launch Agentic Workflows (`Open in Genie Code`):** Directly launch specialized Databricks workflows—such as building scalable document-processing pipelines (parsing, classifying, and extracting structured data from documents at scale).

## Part 7: Top-Right Controls — `Govern` & `Observability`

![part7_1795119516330.png](./part7_1795119516330.png "part7_1795119516330.png")

Across every tab in Unity Gateway, the top-right action bar provides enterprise **Governance** and **Observability** controls:

### 1. `Govern` Dropdown (Policies, Guardrails & Quotas)
* **Service Policies (Guardrails):** Attach real-time input/output checks to any Model Service or MCP Service:
  * **Sensitive Data Detection (`detect_sensitive_data`):** Automatically detects PII/PHI (credit cards, SSNs, emails, secrets) in prompts and responses, and either **blocks** the request or **masks/redacts** the sensitive entities before sending them to the LLM.
  * **Safety & Topic Filters:** Block harmful content or enforce domain boundaries.
* **Rate Limits & Spend Controls:** Configure requests-per-minute (RPM) and tokens-per-minute (TPM) limits per user or endpoint to prevent runaway scripts from exhausting quotas or budgets.
* **Permissions & Lineage:** Manage Unity Catalog `EXECUTE` grants and view downstream lineage showing which apps or agents depend on a given Model Service.

### 2. `Observability` Button (Usage Dashboard & Inference Tables)
* **Built-in Usage Dashboard:** One-click executive and platform dashboard tracking request volume, token consumption, latency, error rates, top users, and estimated USD/DBU spend across all internal and external models.
* **System Tables (`system.ai_gateway.*`):**
  * `system.ai_gateway.usage`: Granular token counts, latency, model routing decisions, and requester identity (`run_by` human vs. `run_as` agent).
  * `system.ai_gateway.external_model_spend`: Hourly aggregated USD cost estimates for third-party model providers.
* **Inference Tables:** Optional payload logging into governed Unity Catalog Delta tables for compliance auditing, prompt debugging, and offline evaluation.

## Part 8: Agent Integrations (Connecting Coding Agents to Unity Gateway)

Notice the **`Agent Integrations`** / **`Connect your coding agents`** cards on the **Models**, **MCPs**, and **Skills** tabs.

### How Developers Connect Local Tools to Unity Gateway
Instead of configuring personal API keys in developer tools, engineers point their coding agents directly at Unity Gateway:
* **Supported Harnesses:** Claude Code, OpenAI Codex CLI, Cursor, VS Code / Continue, and custom Python/TypeScript agents.
* **What Happens on Every Request:**
  1. Developer authenticates using their Databricks identity (`DATABRICKS_TOKEN` / OAuth).
  2. Coding agent sends LLM requests to Unity Gateway's OpenAI-compatible or Anthropic-compatible endpoint.
  3. Coding agent discovers approved **MCP Servers** (`genie_one_mcp`, `dbsql`, `github`, `atlassian`) and **Skills** governed by Unity Catalog.
  4. Platform admins get full per-engineer token attribution, rate limiting, and audit logs in one place.